# Silver utilities

Reusable class for cleaning bronze DataFrames into silver tables.
Load it in another notebook with:

```
%run ./silver_utils
```

In [0]:
from pyspark.sql import DataFrame, SparkSession, Column
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


class SilverTransformer:
    """Chainable cleaning steps for building silver tables.

    Every method returns the transformer itself, so steps can be chained:

        (SilverTransformer.from_table("catalog.bronze.orders")
            .trim()
            .drop_nulls("order_id")
            .deduplicate("order_id")
            .write("catalog.silver.orders"))
    """

    def __init__(self, df: DataFrame):
        self.df = df

    # ---------- create ----------

    @classmethod
    def from_table(cls, table_name: str) -> "SilverTransformer":
        """Start from an existing table, e.g. a bronze table."""
        spark = SparkSession.getActiveSession()
        return cls(spark.read.table(table_name))

    # ---------- cleaning steps ----------

    def trim(self, *columns: str) -> "SilverTransformer":
        """Remove leading/trailing spaces. No columns given = all string columns."""
        if not columns:
            columns = [f.name for f in self.df.schema.fields if isinstance(f.dataType, StringType)]
        for c in columns:
            self.df = self.df.withColumn(c, F.trim(F.col(c)))
        return self

    def drop_nulls(self, *columns: str) -> "SilverTransformer":
        """Drop rows where any of the given columns is null."""
        self.df = self.df.dropna(subset=list(columns))
        return self

    def filter(self, condition) -> "SilverTransformer":
        """Keep rows matching a condition (Column expression or SQL string)."""
        self.df = self.df.filter(condition)
        return self

    def deduplicate(self, *keys: str) -> "SilverTransformer":
        """Keep one row per key. No keys given = remove fully identical rows."""
        self.df = self.df.dropDuplicates(list(keys)) if keys else self.df.dropDuplicates()
        return self

    def cast(self, types: dict) -> "SilverTransformer":
        """Change column types, e.g. {"total_amount": "decimal(12,2)"}."""
        for column, data_type in types.items():
            self.df = self.df.withColumn(column, F.col(column).cast(data_type))
        return self

    def rename(self, mapping: dict) -> "SilverTransformer":
        """Rename columns, e.g. {"cust_id": "customer_id"}."""
        for old, new in mapping.items():
            self.df = self.df.withColumnRenamed(old, new)
        return self

    def add_date_parts(self, date_column: str, prefix: str = None) -> "SilverTransformer":
        """Add <prefix>_year and <prefix>_month columns from a date column."""
        prefix = prefix or date_column
        self.df = (self.df
                   .withColumn(f"{prefix}_year", F.year(date_column))
                   .withColumn(f"{prefix}_month", F.month(date_column)))
        return self

    def add_processed_at(self) -> "SilverTransformer":
        """Add a timestamp showing when the row was processed."""
        self.df = self.df.withColumn("processed_at", F.current_timestamp())
        return self

    def drop(self, *columns: str) -> "SilverTransformer":
        """Remove columns (missing columns are ignored)."""
        self.df = self.df.drop(*columns)
        return self

    def apply(self, func) -> "SilverTransformer":
        """Run any custom step: a function that takes and returns a DataFrame."""
        self.df = func(self.df)
        return self

    # ---------- output ----------

    def to_df(self) -> DataFrame:
        """Return the cleaned DataFrame (e.g. to display it)."""
        return self.df

    def write(self, table_name: str, mode: str = "overwrite") -> DataFrame:
        """Save as a table. Overwrite also allows the column structure to change."""
        writer = self.df.write.mode(mode)
        if mode == "overwrite":
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(table_name)
        print(f"✓ Written to {table_name}")
        return self.df

## Example: silver orders

Uncomment to run here, or copy into your own notebook after `%run ./silver_utils`.

In [0]:
# (SilverTransformer.from_table("databricks_catalog.bronze_schema.orders")
#     .trim("order_id", "customer_id", "product_id")
#     .drop_nulls("order_id", "customer_id", "product_id", "order_date")
#     .filter((F.col("quantity") > 0) & (F.col("total_amount") >= 0))
#     .deduplicate("order_id")
#     .cast({"total_amount": "decimal(12,2)"})
#     .add_date_parts("order_date", prefix="order")
#     .add_processed_at()
#     .drop("_rescued_data")
#     .write("databricks_catalog.silver_schema.orders"))